# 桥水全天候策略与风险平价模型全解析

**大类资产配置量化模型研究系列之三**

本Notebook复现了研报《桥水全天候策略和风险平价模型全解析》中的核心内容，包括：

1. **风险平价模型** - 实现真正的风险分散化投资
2. **波动率倒数模型** - 不考虑相关性的简单风险平价
3. **固定资产比例模型** - 股债商1:8:1固定比例
4. **等权重模型** - 各资产等权配置
5. **夏普预算策略** - 基于夏普率平方的风险预算
6. **加杠杆风险平价** - 通过杠杆提高组合收益
7. **因子风险平价** - 基于主成分的因子风险平价

---

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

print('环境配置完成')

## 1. 数据获取

In [ ]:
from source import fetch_all_assets, save_data, load_data

print('='*60)
print('第一步：获取六类底层资产数据')
print('='*60)

print('\n底层资产：')
print('1. 沪深300指数 (CSI300)')
print('2. 标普500指数 (SPX)')
print('3. 恒生指数 (HSI)')
print('4. 中债-企业债总财富指数 (CBCE)')
print('5. 南华商品指数 (NHCI)')
print('6. COMEX黄金 (GC)')

try:
    prices = load_data('../output/asset_prices.csv')
    if prices.empty:
        raise FileNotFoundError()
    print('已从缓存加载数据')
except:
    print('正在从数据源获取数据...')
    prices = fetch_all_assets('2008-01-01', '2023-04-30')
    save_data(prices, '../output/asset_prices.csv')

print(f'\n数据形状: {prices.shape}')
print(f'时间范围: {prices.index[0].strftime("%Y-%m-%d")} 至 {prices.index[-1].strftime("%Y-%m-%d")}')

In [ ]:
print('\n数据预览:')
prices.head()

In [ ]:
returns = prices.pct_change().dropna()
print(f'收益率数据形状: {returns.shape}')
print('\n各资产年化收益率 (2008-2023):')
annual_returns = (1 + returns.mean())**252 - 1
annual_vol = returns.std() * np.sqrt(252)
print(pd.DataFrame({
    '年化收益': annual_returns.map('{:.2%}'.format),
    '年化波动': annual_vol.map('{:.2%}'.format),
    '夏普比率': ((annual_returns - 0.03) / annual_vol).round(2)
}))

## 2. 资产相关性分析

In [ ]:
import seaborn as sns

corr_matrix = returns.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu_r', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('六类资产相关性矩阵', fontsize=14)
plt.tight_layout()
plt.savefig('../output/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n相关性分析：')
print('企业债与其它资产相关性较低，适合作为底仓配置')

## 3. 运行回测

In [ ]:
from source import BacktestEngine, BacktestConfig

print('='*60)
print('第二步：运行所有策略回测')
print('='*60)

config = BacktestConfig(
    start_date='2009-01-01',
    end_date='2023-04-30',
    lookback_period=126,
    rebalance_freq='M',
    cov_method='sample',
    risk_free_rate=0.03
)

engine = BacktestEngine(prices, config)

print('\n正在运行各策略回测...')
results = engine.run_all_strategies()

print(f'\n共完成 {len(results)} 个策略的回测')

## 4. 绩效评估

In [ ]:
from source import calculate_portfolio_metrics, generate_performance_summary

print('='*60)
print('第三步：绩效评估')
print('='*60)

summary = generate_performance_summary(results, risk_free_rate=0.03)
print('\n策略绩效对比表:')
summary

In [ ]:
from source import calculate_nav, calculate_drawdown_series, calculate_yearly_returns

nav_dict = {}
dd_dict = {}

for name, result in results.items():
    nav_dict[name] = calculate_nav(result.portfolio_returns)
    dd_dict[name] = calculate_drawdown_series(result.portfolio_returns)

nav_df = pd.DataFrame(nav_dict)
dd_df = pd.DataFrame(dd_dict)

print('净值序列预览:')
nav_df.tail()

## 5. 可视化分析

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax1 = axes[0]
for name in nav_df.columns:
    if 'RiskParity' in name or name == 'EqualWeight':
        ax1.plot(nav_df.index, nav_df[name], label=name, linewidth=1.5)
ax1.set_title('净值曲线对比', fontsize=14)
ax1.set_xlabel('日期')
ax1.set_ylabel('净值')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
for name in dd_df.columns:
    if 'RiskParity' in name or name == 'EqualWeight':
        ax2.fill_between(dd_df.index, dd_df[name], 0, alpha=0.3, label=name)
ax2.set_title('回撤曲线对比', fontsize=14)
ax2.set_xlabel('日期')
ax2.set_ylabel('回撤')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/nav_drawdown_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

key_metrics = pd.DataFrame({
    name: {
        '年化收益': calculate_portfolio_metrics(r.result.portfolio_returns)['annualized_return'],
        '最大回撤': calculate_portfolio_metrics(r.result.portfolio_returns)['max_drawdown'],
        '夏普比率': calculate_portfolio_metrics(r.result.portfolio_returns)['sharpe_ratio'],
        '卡玛比率': calculate_portfolio_metrics(r.result.portfolio_returns)['calmar_ratio']
    }
    for name, r in results.items()
}).T

ax1 = axes[0]
x = np.arange(len(key_metrics))
width = 0.2
ax1.bar(x - 1.5*width, key_metrics['年化收益']*100, width, label='年化收益%')
ax1.bar(x - 0.5*width, key_metrics['最大回撤']*100, width, label='最大回撤%')
ax1.bar(x + 0.5*width, key_metrics['夏普比率'], width, label='夏普比率')
ax1.bar(x + 1.5*width, key_metrics['卡玛比率'], width, label='卡玛比率')
ax1.set_xticks(x)
ax1.set_xticklabels(key_metrics.index, rotation=45, ha='right')
ax1.legend()
ax1.set_title('策略绩效指标对比', fontsize=14)
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
colors = plt.cm.Set3(np.linspace(0, 1, len(key_metrics)))
wedges, texts, autotexts = ax2.pie(
    key_metrics['年化收益'],
    labels=key_metrics.index,
    autopct='%1.1f%%',
    colors=colors,
    explode=[0.05]*len(key_metrics)
)
ax2.set_title('年化收益占比', fontsize=14)

plt.tight_layout()
plt.savefig('../output/performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 详细绩效报告

In [ ]:
from source import print_performance_report

for name, result in results.items():
    print_performance_report(name, result.portfolio_returns)
    print()

## 7. 年度收益率对比

In [ ]:
yearly_returns_dict = {}
for name, result in results.items():
    yr = calculate_yearly_returns(result.portfolio_returns)
    yearly_returns_dict[name] = yr

yearly_df = pd.DataFrame(yearly_returns_dict)

print('各策略年度收益率:')
yearly_df.style.format('{:.2%}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

yearly_df.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('各策略年度收益率对比', fontsize=14)
ax.set_xlabel('年份')
ax.set_ylabel('收益率')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../output/yearly_returns.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 权重分析

In [ ]:
rp_weights = results['RiskParity'].weights

print('风险平价策略最新权重:')
latest_weights = rp_weights.iloc[-1]
print(pd.DataFrame({
    '资产': latest_weights.index,
    '权重': latest_weights.values
}).sort_values('权重', ascending=False))

fig, ax = plt.subplots(figsize=(10, 6))
rp_weights.plot(kind='area', stacked=True, ax=ax, alpha=0.7)
ax.set_title('风险平价策略权重演变', fontsize=14)
ax.set_xlabel('日期')
ax.set_ylabel('权重')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig('../output/rp_weights_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. 结论

### 主要发现：

1. **风险平价模型**表现出较高的风险收益比，夏普比率最高
2. **加杠杆风险平价**通过杠杆提高了收益，但最大回撤也相应增加
3. **夏普预算策略**由于增配历史表现好的资产，具有一定动量效应
4. **等权重模型**虽然收益较高，但回撤和波动也最大

### 模型局限性：

- 风险平价放弃了对收益的预测，本质上是"躺平"策略
- 模型在股债双杀时期表现不佳（如2020年新冠疫情期间）
- 建议将风险平价作为战略配置的长期基准，结合战术配置使用

---